In [ ]:
print("Hola desde Jupyter")

Hola desde Jupyter


In [5]:
import os
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# 1. Cargamos las variables de entorno de forma segura
load_dotenv()

try:
    print(f"Intentando conectar a la base de datos de forma segura...")
    
    # 2. Conectamos usando os.getenv() para no mostrar contraseñas en el código
    conexion = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
        connection_timeout=5
    )
    print("¡CONECTADO EXITOSAMENTE REGLAMENTARIO! 🎉\n")
    
    # 3. Validamos las tablas existentes
    cursor = conexion.cursor()
    cursor.execute("SHOW TABLES;")
    tablas = cursor.fetchall()
    
    print(f"Tablas sincronizadas desde '{os.getenv('DB_NAME')}':")
    for t in tablas:
        print(f"  • {t[0]}")
        
    conexion.close()

except mysql.connector.Error as error:
    print(f"\nError de conexión: {error}")
    print("Chequeá que el archivo .env esté bien guardado y en la misma carpeta.")

Intentando conectar a la base de datos de forma segura...
¡CONECTADO EXITOSAMENTE REGLAMENTARIO! 🎉

Tablas sincronizadas desde 'final':
  • autor
  • ejemplar
  • genero
  • libro
  • libro_autor
  • libro_genero
  • prestamo
  • sancion
  • socio
  • tipo_sancion


In [20]:
import os
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# Importamos la función que acabamos de limpiar y guardar en agent.py
from agent import pregunta_a_sql

def ejecutar_agente(pregunta_usuario: str):
    """
    Orquesta el flujo completo: recibe la pregunta, llama al agente para
    obtener el SQL, ejecuta la consulta en MySQL y muestra los resultados.
    """
    # 1. Cargamos las credenciales locales desde el archivo .env
    load_dotenv()
    
    # 2. Generamos el SQL puro llamando a nuestro cerebro en agent.py
    sql_generado = pregunta_a_sql(pregunta_usuario)
    
    print("\n" + "="*50)
    print(f"❓ Pregunta: {pregunta_usuario}")
    print(f"💻 SQL Generado:\n{sql_generado}")
    print("="*50)
    
    # Si la IA devolvió un mensaje de error en vez de un SELECT, frenamos acá
    if sql_generado.startswith("--"):
        print(sql_generado)
        return

    conexion = None
    try:
        # 3. Nos conectamos de forma segura a MySQL usando las variables de entorno
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST", "localhost"),
            port=int(os.getenv("DB_PORT", 3306)),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        
        # 4. Usamos Pandas para ejecutar el SQL y formatear el resultado en una tabla
        df = pd.read_sql(sql_generado, conexion)
        
        print("\n📊 Resultados obtenidos de la base de datos:")
        if df.empty:
            print("No se encontraron registros que cumplan con la condición.")
        else:
            # to_string() hace que se vea como una tablita prolija en la terminal
            print(df.to_string(index=False))
            
    except Exception as e:
        print(f"❌ Error al conectar o ejecutar en la base de datos: {e}")
        
    finally:
        # 5. Cerramos la conexión siempre para no dejar hilos colgados en MySQL
        if conexion and conexion.is_connected():
            conexion.close()

# --- BLOQUE DE EJECUCIÓN PRINCIPAL ---
if __name__ == "__main__":
    # Una prueba rápida al ejecutar: 'python main.py' en la terminal
    consulta_prueba = "¿Cuántos libros hay de cada género en la biblioteca?"
    ejecutar_agente(consulta_prueba)


❓ Pregunta: ¿Cuántos libros hay de cada género en la biblioteca?
💻 SQL Generado:
SELECT g.nombre, COUNT(l.isbn) 
FROM libro l 
JOIN libro_genero lg ON l.isbn = lg.isbn 
JOIN genero g ON lg.id_genero = g.id_genero 
GROUP BY g.nombre

📊 Resultados obtenidos de la base de datos:
                nombre  COUNT(l.isbn)
       Ciencia Ficción              5
Divulgación Científica              4
              Fantasía              5
      Novela Histórica              3
                Terror              2


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\601135492.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql_generado, conexion)


In [21]:
import os
import mysql.connector
import pandas as pd

def agente_responder(pregunta_usuario):
    print(f"❓ Pregunta del usuario: {pregunta_usuario}")
    
    try:
        # 1. Le pedimos el SQL a la IA
        sql = consultar_ia(pregunta_usuario)
        print(f"💻 SQL Generado: {sql}\n")
        
        # 2. Conectamos a MySQL
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=int(os.getenv("DB_PORT")),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        
        # 3. Ejecutamos y pasamos a DataFrame
        df = pd.read_sql(sql, conexion)
        conexion.close()
        
        if not df.empty:
            print("📊 Resultados obtenidos de la base de datos:")
            return df
        else:
            print("⚠ La consulta se ejecutó bien, pero no devolvió ningún registro.")
            return None
            
    except Exception as error:
        print(f"❌ Error al procesar la solicitud: {error}")
        return None

# --- PRUEBA DE FUEGO INTEGRADORA ---
agente_responder("¿Cuáles son los títulos de los libros que tenemos en la biblioteca?")

❓ Pregunta del usuario: ¿Cuáles son los títulos de los libros que tenemos en la biblioteca?
💻 SQL Generado: SELECT titulo FROM libro

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,Y no quedó ninguno
1,Asesinato en el Orient Express
2,Sapiens: De animales a dioses
3,El Señor de los Anillos: La Comunidad del Anillo
4,El Señor de los Anillos: Las Dos Torres
5,El Señor de los Anillos: El Retorno del Rey
6,Cien años de soledad
7,El Resplandor
8,Ficciones
9,Cosmos


In [22]:
agente_responder("Mostrame los títulos de los libros escritos por el autor Isaac Asimov")

❓ Pregunta del usuario: Mostrame los títulos de los libros escritos por el autor Isaac Asimov
💻 SQL Generado: SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_autor la ON l.isbn = la.isbn 
JOIN autor a ON la.id_autor = a.id_autor 
WHERE a.nombre = 'Isaac' AND a.apellido = 'Asimov'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,Fundación
1,Fundación e Imperio


In [23]:
agente_responder("¿Qué socios tienen préstamos vencidos actualmente? Mostrame sus nombres y apellidos")

❓ Pregunta del usuario: ¿Qué socios tienen préstamos vencidos actualmente? Mostrame sus nombres y apellidos
💻 SQL Generado: SELECT DISTINCT s.nombre, s.apellido 
FROM socio s 
JOIN prestamo p ON s.id_socio = p.id_socio 
WHERE p.estado = 'VENCIDO'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,nombre,apellido
0,Juan,Rodríguez
1,Valentina,Álvarez
2,Mateo,Romero
3,Emma,Sánchez


In [24]:
# ====================================================================
# DEMOSTRACIÓN FINAL DEL AGENTE (PRUEBAS OBLIGATORIAS DEL TP)
# ====================================================================

print("--- PRUEBA 1: Búsqueda con Joins y agregación ---")
display(agente_responder("¿Qué libros escribió el autor J.R.R. Tolkien?"))
print("-" * 60)

print("\n--- PRUEBA 2: Filtros de estados y fechas ---")
display(agente_responder("Mostrame los socios que están suspendidos"))
print("-" * 60)

print("\n--- PRUEBA 3: Funciones de agregación y ordenamiento ---")
display(agente_responder("¿Cuáles son los 3 libros con mayor stock total en la biblioteca?"))
print("-" * 60)

print("\n--- PRUEBA 4: Cruce complejo de tres tablas ---")
display(agente_responder("Listá los títulos de los libros que pertenecen al género Fantasía"))
print("-" * 60)

--- PRUEBA 1: Búsqueda con Joins y agregación ---
❓ Pregunta del usuario: ¿Qué libros escribió el autor J.R.R. Tolkien?
💻 SQL Generado: SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_autor la ON l.isbn = la.isbn 
JOIN autor a ON la.id_autor = a.id_autor 
WHERE a.nombre = 'J.R.R.' AND a.apellido = 'Tolkien'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,El Señor de los Anillos: La Comunidad del Anillo
1,El Señor de los Anillos: Las Dos Torres
2,El Señor de los Anillos: El Retorno del Rey


------------------------------------------------------------

--- PRUEBA 2: Filtros de estados y fechas ---
❓ Pregunta del usuario: Mostrame los socios que están suspendidos
💻 SQL Generado: SELECT * FROM socio WHERE estado = 'SUSPENDIDO'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


...

,titulo,stock_total
0,1984,5
1,El Señor de los Anillos: La Comunidad del Anillo,5
2,Sapiens: De animales a dioses,4


------------------------------------------------------------

--- PRUEBA 4: Cruce complejo de tres tablas ---
❓ Pregunta del usuario: Listá los títulos de los libros que pertenecen al género Fantasía
💻 SQL Generado: SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_genero lg ON l.isbn = lg.isbn 
JOIN genero g ON lg.id_genero = g.id_genero 
WHERE g.nombre = 'Fantasía'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,El Señor de los Anillos: La Comunidad del Anillo
1,El Señor de los Anillos: Las Dos Torres
2,El Señor de los Anillos: El Retorno del Rey
3,Juego de Tronos
4,Choque de Reyes


------------------------------------------------------------


In [25]:
agente_responder("¿Cuántos libros hay de cada género en la biblioteca?")

❓ Pregunta del usuario: ¿Cuántos libros hay de cada género en la biblioteca?
💻 SQL Generado: SELECT g.nombre, COUNT(lg.isbn) 
FROM genero g 
JOIN libro_genero lg ON g.id_genero = lg.id_genero 
GROUP BY g.nombre

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,nombre,COUNT(lg.isbn)
0,Ciencia Ficción,5
1,Divulgación Científica,4
2,Fantasía,5
3,Novela Histórica,3
4,Terror,2


In [28]:
agente_responder("Aquellos libros para los que existe más de un ejemplar, tal que al menos dos de esos ejemplares se hayan encontrado prestados en forma simultánea en un determinado momento. Para simplificar, considerar solamente aquellos préstamos en los que el libro ya haya sido devuelto.")

❓ Pregunta del usuario: Aquellos libros para los que existe más de un ejemplar, tal que al menos dos de esos ejemplares se hayan encontrado prestados en forma simultánea en un determinado momento. Para simplificar, considerar solamente aquellos préstamos en los que el libro ya haya sido devuelto.
💻 SQL Generado: SELECT DISTINCT p1.id_ejemplar, l.titulo 
FROM prestamo p1 
JOIN prestamo p2 ON p1.id_ejemplar <> p2.id_ejemplar AND p1.id_ejemplar IN (SELECT id_ejemplar FROM ejemplar WHERE isbn = (SELECT isbn FROM ejemplar WHERE id_ejemplar = p2.id_ejemplar)) 
JOIN ejemplar e1 ON p1.id_ejemplar = e1.id_ejemplar 
JOIN ejemplar e2 ON p2.id_ejemplar = e2.id_ejemplar 
JOIN libro l ON e1.isbn = l.isbn 
WHERE p1.fecha_prestamo <= p2.fecha_devolucion AND p1.fecha_devolucion >= p2.fecha_prestamo AND p1.fecha_devolucion IS NOT NULL AND p2.fecha_devolucion IS NOT NULL 
AND l.isbn IN (SELECT isbn FROM ejemplar GROUP BY isbn HAVING COUNT(id_ejemplar) > 1)

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_23384\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,id_ejemplar,titulo
0,7,El Señor de los Anillos: La Comunidad del Anillo
1,6,El Señor de los Anillos: La Comunidad del Anillo
2,8,El Señor de los Anillos: La Comunidad del Anillo
3,5,2001: Una Odisea Espacial
4,4,2001: Una Odisea Espacial
5,10,Juego de Tronos
6,9,Juego de Tronos
7,2,Fundación
8,3,Fundación
9,1,Fundación
